In [13]:
import numpy as np
np.set_printoptions(precision=4, suppress=True)


In [14]:
# ============ DESAFIO 1 — PERCEPTRON (FRAUDE) ============
X_treino_fraude = np.array([
    [1.5, 1.0],
    [2.0, 2.0],
    [3.5, 1.5],
    [3.0, 3.0],
    [6.5, 5.0],
    [7.0, 7.0],
    [8.5, 6.0],
    [9.0, 8.0]
])

y_treino_fraude = np.array([0, 0, 0, 0, 1, 1, 1, 1])


In [15]:
def treinar_perceptron(X, y, eta=0.01, epocas=100):
    """
    Treina um Perceptron com a regra de aprendizado de Rosenblatt.
    - Pesos e viés inicializados com zero.
    - Atualiza apenas quando há erro de predição.
    Retorna (w, b).
    """
    n_amostras, n_features = X.shape
    w = np.zeros(n_features)
    b = 0.0

    for epoca in range(epocas):
        erros = 0
        for i in range(n_amostras):
            saida = np.dot(X[i], w) + b
            y_pred = 1 if saida >= 0 else 0

            erro = y[i] - y_pred
            if erro != 0:
                w += eta * erro * X[i]
                b += eta * erro
                erros += 1

        if erros == 0:
            print(f"Convergiu na época {epoca + 1}.")
            break
    else:
        print(f"Não convergiu em {epocas} épocas.")

    return w, b


In [16]:
def prever_perceptron(x, w, b):
    """Retorna 0 ou 1 aplicando a função degrau."""
    return 1 if (np.dot(x, w) + b) >= 0 else 0


In [17]:
w_fraude, b_fraude = treinar_perceptron(
    X_treino_fraude, y_treino_fraude, eta=0.01, epocas=100
)

print("\n" + "=" * 60)
print("DESAFIO 1 — PERCEPTRON (FRAUDE)")
print("=" * 60)
print(f"Pesos finais (w) : {w_fraude}")
print(f"Viés final  (b)  : {b_fraude:.4f}")
print(f"Fronteira: {w_fraude[0]:.3f}*x1 + {w_fraude[1]:.3f}*x2 + ({b_fraude:.3f}) = 0")

transacoes = {
    "Transação A": np.array([2.5, 2.0]),
    "Transação B": np.array([8.0, 6.5]),
}

for nome, x in transacoes.items():
    classe = prever_perceptron(x, w_fraude, b_fraude)
    rotulo = "Suspeita (Risco de Fraude)" if classe == 1 else "Legítima"
    score = np.dot(x, w_fraude) + b_fraude
    print(f"\n{nome}: {x.tolist()}")
    print(f"  Score (w·x + b) = {score:.4f}")
    print(f"  Classe predita  = {classe} → {rotulo}")


Convergiu na época 6.

DESAFIO 1 — PERCEPTRON (FRAUDE)
Pesos finais (w) : [0.005 0.01 ]
Viés final  (b)  : -0.0800
Fronteira: 0.005*x1 + 0.010*x2 + (-0.080) = 0

Transação A: [2.5, 2.0]
  Score (w·x + b) = -0.0475
  Classe predita  = 0 → Legítima

Transação B: [8.0, 6.5]
  Score (w·x + b) = 0.0250
  Classe predita  = 1 → Suspeita (Risco de Fraude)


In [18]:
# ============ DESAFIO 2 — KNN (CHURN) ============
X_treino_churn = np.array([
    [2.0, 1.0],
    [3.0, 0.0],
    [5.0, 1.0],
    [6.0, 2.0],
    [15.0, 4.0],
    [18.0, 5.0],
    [20.0, 4.0],
    [22.0, 6.0]
])

y_treino_churn = np.array([0, 0, 0, 0, 1, 1, 1, 1])


In [19]:
def knn_classificar(X, y, ponto, K=3, metrica="euclidiana"):
    """
    Classifica 'ponto' pelo KNN com cálculo vetorizado das distâncias.
    Retorna (classe_predita, indices_vizinhos, distancias_vizinhos).
    """
    ponto = np.asarray(ponto, dtype=float)
    dif = X - ponto  # broadcasting

    if metrica.lower() == "euclidiana":
        distancias = np.sqrt(np.sum(dif ** 2, axis=1))
    elif metrica.lower() == "manhattan":
        distancias = np.sum(np.abs(dif), axis=1)
    else:
        raise ValueError("Métrica inválida. Use 'euclidiana' ou 'manhattan'.")

    indices_ordenados = np.argsort(distancias)
    indices_vizinhos = indices_ordenados[:K]
    distancias_vizinhos = distancias[indices_vizinhos]

    classes_vizinhos = y[indices_vizinhos]
    contagem = np.bincount(classes_vizinhos)
    classe_predita = int(np.argmax(contagem))

    return classe_predita, indices_vizinhos, distancias_vizinhos


In [20]:
print("\n" + "=" * 60)
print("DESAFIO 2 — KNN (CHURN)")
print("=" * 60)

clientes = {
    "Cliente 1": np.array([4.0, 1.0]),
    "Cliente 2": np.array([17.0, 5.0]),
}

for nome, ponto in clientes.items():
    classe, idx, dists = knn_classificar(
        X_treino_churn, y_treino_churn, ponto, K=3, metrica="euclidiana"
    )
    rotulo = "Alto Risco" if classe == 1 else "Baixo Risco"
    print(f"\n{nome}: {ponto.tolist()} | K=3 | Euclidiana")
    print(f"  Classe predita      : {classe} → {rotulo}")
    print(f"  Índices dos vizinhos: {idx.tolist()}")
    print(f"  Distâncias          : {np.round(dists, 4).tolist()}")



DESAFIO 2 — KNN (CHURN)

Cliente 1: [4.0, 1.0] | K=3 | Euclidiana
  Classe predita      : 0 → Baixo Risco
  Índices dos vizinhos: [2, 1, 0]
  Distâncias          : [1.0, 1.4142, 2.0]

Cliente 2: [17.0, 5.0] | K=3 | Euclidiana
  Classe predita      : 1 → Alto Risco
  Índices dos vizinhos: [5, 4, 6]
  Distâncias          : [1.0, 2.2361, 3.1623]


In [21]:
print("\n--- Verificação extra: Cliente 1 com Manhattan ---")
classe, idx, dists = knn_classificar(
    X_treino_churn, y_treino_churn, clientes["Cliente 1"],
    K=3, metrica="manhattan"
)
rotulo = "Alto Risco" if classe == 1 else "Baixo Risco"
print(f"  Classe: {classe} → {rotulo}")
print(f"  Vizinhos: {idx.tolist()}")
print(f"  Distâncias: {np.round(dists, 4).tolist()}")



--- Verificação extra: Cliente 1 com Manhattan ---
  Classe: 0 → Baixo Risco
  Vizinhos: [2, 0, 1]
  Distâncias: [1.0, 2.0, 2.0]


In [22]:
# ============ DESAFIO 3 — SERVIDORES CLOUD ============
catalogo_servidores = np.array([
    [2.0, 4.0, 50.0],
    [4.0, 8.0, 100.0],
    [8.0, 16.0, 250.0],
    [16.0, 32.0, 500.0],
    [32.0, 64.0, 1000.0],
    [64.0, 128.0, 2000.0]
])

nomes_servidores = [
    "Micro Instância Web",
    "Standard App Server",
    "Medium Backend & Cache",
    "Database Enterprise",
    "High Performance Computing",
    "Big Data & AI Training"
]


In [23]:
def recomendar_servidores(catalogo, nomes, perfil, K=2):
    """Recomenda K servidores mais similares ao perfil (Euclidiana vetorizada)."""
    perfil = np.asarray(perfil, dtype=float)
    dif = catalogo - perfil
    distancias = np.sqrt(np.sum(dif ** 2, axis=1))

    indices_ordenados = np.argsort(distancias)[:K]

    recomendacoes = []
    for pos, idx in enumerate(indices_ordenados, start=1):
        recomendacoes.append((pos, nomes[idx], catalogo[idx], distancias[idx]))
    return recomendacoes


In [24]:
print("\n" + "=" * 60)
print("DESAFIO 3 — RECOMENDAÇÃO DE SERVIDORES CLOUD")
print("=" * 60)

perfil_demanda = np.array([12.0, 28.0, 450.0])
K = 2

recs = recomendar_servidores(
    catalogo_servidores, nomes_servidores, perfil_demanda, K=K
)

print(f"Perfil demandado: vCPUs={perfil_demanda[0]:.0f} | "
      f"RAM={perfil_demanda[1]:.0f} GB | SSD={perfil_demanda[2]:.0f} GB")
print(f"K = {K}\n")

for pos, nome, specs, dist in recs:
    print(f"{pos}º lugar — {nome}")
    print(f"   vCPUs={specs[0]:.0f} | RAM={specs[1]:.0f} GB | SSD={specs[2]:.0f} GB")
    print(f"   Distância euclidiana até a demanda: {dist:.4f}\n")



DESAFIO 3 — RECOMENDAÇÃO DE SERVIDORES CLOUD
Perfil demandado: vCPUs=12 | RAM=28 GB | SSD=450 GB
K = 2

1º lugar — Database Enterprise
   vCPUs=16 | RAM=32 GB | SSD=500 GB
   Distância euclidiana até a demanda: 50.3190

2º lugar — Medium Backend & Cache
   vCPUs=8 | RAM=16 GB | SSD=250 GB
   Distância euclidiana até a demanda: 200.3996

